In [ ]:
# IMPORTS

from src_sequel.classes.single_encoder import SingleEncoder
from classes.mistral_7b import MistralGenerator
from helpers.generate_embeddings import generate_embeddings
from helpers.load_embeddings import load_embeddings
from helpers.build_prompt import build_prompt
from helpers.retrieve_top_k import retrieve_top_k
from transformers import CanineModel, CanineTokenizer
import pandas as pd
import torch
import faiss
import numpy as np
import pickle
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Skipping import of cpp extensions due to incompatible torch version 2.7.1+cu118 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0114 13:12:12.014000 12240 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


ModuleNotFoundError: No module named 'rag'

In [ ]:
# SETUP
MODEL_PATH = "../output/models/"
EMBEDDINGS_PATH = "../output/embeddings/"
CHAT_DATA_PATH = "../data/processed/"

MODEL_NAME = "style_retriever_author_contrastive"
CHAT_DATA = "private_full"


MODEL_PATH = MODEL_PATH + MODEL_NAME
EMBEDDINGS_PATH = EMBEDDINGS_PATH + CHAT_DATA
CHAT_DATA_PATH = CHAT_DATA_PATH + CHAT_DATA

In [ ]:
# LOAD MODEL
tokenizer = CanineTokenizer.from_pretrained(MODEL_PATH)
encoder = CanineModel.from_pretrained(MODEL_PATH)

model = SingleEncoder()
model.encoder = encoder
model.proj.load_state_dict(torch.load(f"{MODEL_PATH}/projection_head.pt"))
model.eval()
model.to("cuda")

In [ ]:
# LOAD CHAT DATA AND GENERATE EMBEDDINGS
df = pd.read_csv(CHAT_DATA_PATH)
df = df.dropna(subset=["Content"])
df["Content"] = df["Content"].astype(str)

print("Messages:", len(df))
print("Authors:", df["Author"].nunique())

texts = df["Content"].tolist()

embeddings = generate_embeddings(
    texts=texts,
    model=model,
    tokenizer=tokenizer,
    batch_size=8,
    device=device,
)

print("Embedding shape:", embeddings.shape)

records = [
    {
        "author": df.iloc[i]["Author"],
        "content": df.iloc[i]["Content"]
    }
    for i in range(len(df))
]

# Save to FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # cosine similarity
index.add(embeddings.numpy())

print("Vectors in index:", index.ntotal)

os.makedirs(EMBEDDINGS_PATH, exist_ok=True)

faiss.write_index(index, os.path.join(EMBEDDINGS_PATH, "style_index.faiss"))
with open(os.path.join(EMBEDDINGS_PATH, "style_metadata.pkl"), "wb") as f:
    pickle.dump(records, f)
torch.save(embeddings, os.path.join(EMBEDDINGS_PATH, "style_embeddings.pt"))

In [ ]:
# load embeddings directly if already generated
index, records, embeddings = load_embeddings(EMBEDDINGS_PATH)

In [ ]:
generator = MistralGenerator()

In [ ]:
# Example incoming message
incoming_message = "lol that was actually wild"

# Retrieve style examples
style_examples = retrieve_top_k(
    query_text=incoming_message,
    model=model,
    tokenizer=tokenizer,
    index=index,
    records=records,
    device=device,
    k=5
)

# Build prompt
prompt = build_prompt(incoming_message, style_examples)
print(prompt)
print("==================")

# Generate response suggestion

response = generator.generate(prompt)

print(response)